# Step 06 — Four search algorithms on London road network


In [1]:
import sys
from pathlib import Path

# Check Python version
if sys.version_info < (3, 11):

    raise RuntimeError(
        f"Python 3.11+ required. "
        f"Current version: {sys.version}"
    )

# Check package installation
import routing

print("Python version:", sys.version)
print("Python kernel:", sys.executable)
print("Routing package:", Path(routing.__file__).resolve())

print("Environment check: PASSED")

Python version: 3.14.8 (main, Sep 30 2026, 17:55:09) [Clang 21.0.0 (clang-2100.3.34.2)]
Python kernel: /Users/mojtabashabani/wiser/mary/.venv/bin/python
Routing package: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/src/routing/__init__.py
Environment check: PASSED


In [2]:
import sys
import pandas as pd
from IPython.display import display

from routing.config import ORIGIN, ORIGINS, DESTINATION
from routing.map_loader import load_road_graph, nearest_drive_node
from routing.map_view import interactive_map
from routing.traffic_pipeline import collect_traffic_snapshot, make_routing_graph
from routing.algorithms import bfs, dfs, greedy_best_first, astar, run_all_algorithms
from routing.route_view import add_route_overlay

print("Python kernel:", sys.executable)
RADIUS_M = 2000
G = load_road_graph(radius_m=RADIUS_M)
print("OSM graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "directed edges")
assert G.is_directed() and G.is_multigraph()

Python kernel: /Users/mojtabashabani/wiser/mary/.venv/bin/python
OSM graph: 2569 nodes, 5850 directed edges


## 2 — Prepare one shared, safe offline snapshot

In [3]:


snapshot, summary = collect_traffic_snapshot(G, None, max_requests=0)
assert summary["requests_made"] == 0
assert summary["usable_observed_edges"] == 0

measured_time = make_routing_graph(snapshot, objective="time", scenario="measured")
measured_distance = make_routing_graph(snapshot, objective="distance", scenario="measured")
free_flow_time = make_routing_graph(snapshot, objective="time", scenario="free_flow")

print("API requests:", summary["requests_made"])
print("Cost attribute (time):", measured_time.graph["routing_weight"])
print("Cost attribute (distance):", measured_distance.graph["routing_weight"])
print("NOTICE: offline measured travel times are IMPUTED, not actual traffic observations.")

API requests: 0
Cost attribute (time): travel_time_s
Cost attribute (distance): length
NOTICE: offline measured travel times are IMPUTED, not actual traffic observations.


## 3 — Choose your origin by double-clicking on the map



In [4]:
route_widget, route_state = interactive_map(
    G, origin=ORIGIN, destination=DESTINATION, traffic_graph=snapshot,
    max_snap_distance_m=350.0,
)
display(route_widget)

## 4 — Capture the selected nodes

In [5]:
source = route_state["origin_node"]
goal = route_state["destination_node"]
print("Origin node:", source)
print("Destination node:", goal)
print("Last double-click:", route_state["last_click_latlon"])
assert source in G and goal in G



Origin node: 25291778
Destination node: 26389375
Last double-click: None


## 5 — Run all four algorithms (time objective)

In [6]:
results_time = run_all_algorithms(measured_time, source, goal)

cols = ["algorithm", "found", "hops", "distance_m", "travel_time_s",
        "objective_cost", "imputed_route_edges", "expanded_nodes",
        "max_frontier_size", "runtime_ms"]
rows = [{k: result[k] for k in cols} for result in results_time.values()]
display(pd.DataFrame(rows).round(3))

if not results_time["A*"]["found"]:
    print("No directed path between these selected nodes. Choose another road for the origin.")
else:
    print("A* time-optimal path:", results_time["A*"]["path_edges"][:8], "...")

,algorithm,found,hops,distance_m,travel_time_s,objective_cost,imputed_route_edges,expanded_nodes,max_frontier_size,runtime_ms
0,bfs,True,5,679.322,81.519,81.519,5,44,18,0.177
1,DFS,True,209,15561.308,1867.357,1867.357,209,610,162,1.888
2,Greedy,True,5,679.322,81.519,81.519,5,6,8,3.834
3,A*,True,6,628.181,75.382,75.382,6,9,10,3.561


A* time-optimal path: [(25291778, 1360237602, 0), (1360237602, 20963439, 0), (20963439, 6280192965, 0), (6280192965, 6280192964, 0), (6280192964, 34519894, 0), (34519894, 26389375, 0)] ...


## 6 — Overlay the four paths on the SAME working JupyterLab map



In [7]:
for name, result in results_time.items():
    if result["found"]:
        add_route_overlay(route_widget, G, result, name=f"{name} | time")
        print(name, "route drawn; edges:", len(result["path_edges"]))
    else:
        print(name, "no route available")
print("Routes were added to the map displayed above.")

BFS route drawn; edges: 5
DFS route drawn; edges: 209
Greedy route drawn; edges: 5
A* route drawn; edges: 6
Routes were added to the map displayed above.


## 7 — Change objective: distance (same source, goal, and snapshot)

In [8]:
results_distance = run_all_algorithms(measured_distance, source, goal)
display(pd.DataFrame([
    {"algorithm": name, "found": r["found"], "distance_m": r["distance_m"],
     "travel_time_s": r["travel_time_s"], "expanded_nodes": r["expanded_nodes"],
     "runtime_ms": r["runtime_ms"]}
    for name, r in results_distance.items()
]).round(3))

if results_distance["A*"]["found"]:
    print("A* minimum distance:", results_distance["A*"]["distance_m"], "m")
    

,algorithm,found,distance_m,travel_time_s,expanded_nodes,runtime_ms
0,BFS,True,679.322,81.519,44,0.155
1,DFS,True,15561.308,1867.357,610,1.944
2,Greedy,True,679.322,81.519,6,3.859
3,A*,True,628.181,75.382,9,3.900


A* minimum distance: 628.1810704401136 m


## 8 — Free-flow time scenario

In [9]:
results_free = run_all_algorithms(free_flow_time, source, goal)
for name, result in results_free.items():
    print(name, "found:", result["found"],
          "free-flow seconds:", result["travel_time_s"])


BFS found: True free-flow seconds: 81.51868933136596
DFS found: True free-flow seconds: 1867.3569547102102
Greedy found: True free-flow seconds: 81.51868933136596
A* found: True free-flow seconds: 75.38172845281363


## 9 — Road closure scenario: remove an edge from the A* route



In [10]:
from routing.traffic_pipeline import make_routing_graph

best_route = results_time["A*"]
if best_route["found"] and best_route["path_edges"]:
    chosen_closure = best_route["path_edges"][len(best_route["path_edges"]) // 2]
    closure_graph = make_routing_graph(
        snapshot, objective="time", scenario="closure",
        imposed_closure=chosen_closure,
    )
    assert not closure_graph.has_edge(*chosen_closure)
    closure_results = run_all_algorithms(closure_graph, source, goal)
    print("Imposed closure:", chosen_closure)
    display(pd.DataFrame([
        {"algorithm": name, "found": r["found"],
         "travel_time_s": r["travel_time_s"], "hops": r["hops"]}
        for name, r in closure_results.items()
    ]).round(3))
else:
    print("No nonempty A* path available for a closure demonstration.")

Imposed closure: (6280192965, 6280192964, 0)


,algorithm,found,travel_time_s,hops
0,BFS,True,81.519,5
1,DFS,True,1867.357,209
2,Greedy,True,81.519,5
3,A*,True,81.519,5


## 10 — Automatic smoke checks (NO NetworkX shortest-path used in implementations)

In [11]:
for results in (results_time, results_distance, results_free):
    for name, result in results.items():
        if result["found"]:
            assert result["path_nodes"][0] == source
            assert result["path_nodes"][-1] == goal
            assert all(G.has_edge(*edge) for edge in result["path_edges"])
            assert len(result["path_edges"]) == len(result["path_nodes"]) - 1
print("PASS: Four algorithms have valid route structures across three graph objectives/scenarios")
print("Reminder: realistic traffic-aware comparison needs a quality-checked live snapshot.")

PASS: Four algorithms have valid route structures across three graph objectives/scenarios
Reminder: realistic traffic-aware comparison needs a quality-checked live snapshot.


In [12]:
source = route_state["origin_node"]
goal = route_state["destination_node"]

print("Origin:", source)
print("Destination:", goal)
print("Last double-click:", route_state["last_click_latlon"])

assert source in G
assert goal in G

assert route_state["last_click_latlon"] is not None

print("PASS: Interactive origin selection")

Origin: 25291778
Destination: 26389375
Last double-click: None


AssertionError: 